# YOLO12-Small + EMA-32 - CH-RDD2022 (Kaggle)

Notebook ini menjalankan YOLO12s + Efficient Multi-scale Attention (EMA-32) saja. EMA factor 32 dipasang pada P3/8; tidak ada GhostConv, ECA, atau SPD-Conv.

Notebook meng-clone branch ini, mentransfer tensor kompatibel dari yolo12s.pt, mencetak model, melatih, mengevaluasi validation/test, lalu membuat ZIP hasil. Aktifkan GPU dan Internet di Kaggle.

In [ ]:
# 1. Clone branch modifikasi dan install repository sebagai source yang dipakai training.
import json
import platform
import re
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ema32'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))
import torch
import ultralytics

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Commit      : {REPO_COMMIT}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Konfigurasi data/hyperparameter dan print model info sebelum training.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ema32.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML,
)

# Batch 16 sesuai kapasitas GPU; nbs=64 mempertahankan nominal batch untuk gradient accumulation.
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED, EMA_FACTOR = 0, 2, 42, 32
EXPERIMENT_NAME = 'yolo12s_ema32_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists(), f'Model YAML tidak ditemukan: {MODEL_YAML}'
assert all(path.exists() for path in CUSTOM_SOURCE_FILES), 'Source EMA-32 tidak lengkap.'

from ultralytics.nn.modules import EMAAttention, GhostConv
from ultralytics.nn.tasks import DetectionModel

log_section('YOLO12S + EMA-32 MODEL INFO')
print(MODEL_YAML.read_text(encoding='utf-8'))
check_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=True)
parameter_count = sum(parameter.numel() for parameter in check_model.parameters())
ema_layers = [(layer.i, layer.groups) for layer in check_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
assert ema_layers == [(5, EMA_FACTOR)], f'EMA-32 tidak terbentuk sesuai konfigurasi: {ema_layers}'
assert not ghost_layers, f'Varian EMA-32 murni tidak boleh mempunyai GhostConv: {ghost_layers}'
check_model.eval()
with torch.inference_mode():
    model_output = check_model(torch.zeros(1, 3, IMGSZ, IMGSZ))
assert isinstance(model_output, tuple) and model_output[0].shape[1] == 9
print(f'Parameters (5 classes): {parameter_count:,}')
print(f'EMA layers           : {ema_layers}')
print(f'GhostConv layers     : {ghost_layers} (expected empty)')
check_model.info(detailed=False, verbose=True)
del check_model, model_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 3. Transfer pretrained YOLO12s. EMA baru tetap diinisialisasi dan dipelajari saat fine-tuning.
from ultralytics import YOLO

PRETRAINED_WEIGHTS = 'yolo12s.pt'

def target_layer_index(source_index: int) -> int:
    # EMA disisipkan setelah source backbone layer 4, sehingga layer asli 5 dan seterusnya bergeser satu indeks.
    return source_index + int(source_index >= 5)

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'policy': 'remap all compatible YOLO12s tensors around EMA insertion after source layer 4',
    'transferred_tensors': len(transferred_state), 'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
    'uninitialized_tensor_names': incompatible.missing_keys,
}
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Transferred tensors : {PRETRAINED_REPORT['transferred_tensors']}/{PRETRAINED_REPORT['target_tensors']}")
print(f"Uninitialized tensors: {PRETRAINED_REPORT['uninitialized_tensors']} (EMA-32 dan head 5 kelas)")
print('Model akan fine-tuning dari yolo12s.pt, bukan training dari nol.')


In [ ]:
# 4. Training. Pertahankan setting ini untuk perbandingan ablation yang fair.
log_section('TRAINING STARTED - YOLO12S + EMA-32')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
    seed=SEED, plots=True, verbose=True,
)
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')


In [ ]:
# 5. Evaluasi best.pt dan buat ZIP hasil yang siap diunduh dari Kaggle Output.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                    conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                    name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML), 'ema_factor': EMA_FACTOR,
    'ghost_conv_positions': [], 'pretrained_transfer': PRETRAINED_REPORT,
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER, 'lr0': LR0,
    'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')
print(json.dumps(EVALUATION_REPORT, indent=2))

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    file_count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))
print(f'ZIP created : {ZIP_PATH}')
print(f'Files added : {file_count}')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
